# BigMart Outlet Sales Prediction
### End-to-end regression project

This notebook follows the workflow used in the Boston data notebook: load data, inspect data types and missing values, explore distributions/relationships, prepare features, train models, evaluate with RMSE, and create a submission file.

**Important:** only `train.csv` was attached in the chat. The notebook will also load `test.csv` if you place it in the same folder. It cannot create the final test predictions without that file.


In [ ]:
# If using Google Colab, uncomment the next lines to upload train.csv and test.csv
# from google.colab import files
# uploaded = files.upload()

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from catboost import CatBoostRegressor

pd.set_option("display.max_columns", None)
sns.set_theme()


In [ ]:
# Load training data
train_path = "train.csv"
df = pd.read_csv(train_path)

print("Dataset shape:", df.shape)
display(df.head())
df.info()


In [ ]:
# Check missing values and duplicate rows
print("Missing values per column:")
display(df.isnull().sum().sort_values(ascending=False))
print("Duplicate rows:", df.duplicated().sum())

target_var = "Item_Outlet_Sales"
independent_vars = [col for col in df.columns if col != target_var]
print("\nTarget variable:", target_var)
print("Number of features:", len(independent_vars))


In [ ]:
# Summary statistics
display(df.describe(include="all").T)

# Target distribution
plt.figure(figsize=(9, 5))
sns.histplot(df[target_var], bins=40, kde=True)
plt.title("Distribution of Item Outlet Sales")
plt.show()


In [ ]:
# Numerical feature distributions
df.select_dtypes(include=np.number).hist(figsize=(14, 10), bins=25, edgecolor="black")
plt.suptitle("Numerical Feature Distributions", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Sales by outlet type and item type
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x="Outlet_Type", y=target_var)
plt.xticks(rotation=20)
plt.title("Sales by Outlet Type")
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
sns.barplot(data=df, x="Item_Type", y=target_var, errorbar=None)
plt.xticks(rotation=70, ha="right")
plt.title("Average Sales by Item Type")
plt.tight_layout()
plt.show()


In [ ]:
# Correlation matrix for numeric variables
plt.figure(figsize=(10, 7))
sns.heatmap(df.select_dtypes(include=np.number).corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()


## Data preparation and feature engineering

- Standardize inconsistent labels in `Item_Fat_Content`.
- Fill missing item weights using the median weight for the same item, then the overall median.
- Keep missing outlet size as an explicit `"Unknown"` category.
- Derive outlet age from the establishment year and item identifier prefix.
- Add visibility relative to the outlet's average visibility.
- Keep item and outlet identifiers available as categorical signals.

The transformations are applied separately to the training and validation/test partitions where appropriate. Tree models do not require feature scaling.


In [ ]:
def prepare_features(data):
    data = data.copy()

    # Clean inconsistent labels
    data["Item_Fat_Content"] = data["Item_Fat_Content"].replace({
        "LF": "Low Fat",
        "low fat": "Low Fat",
        "reg": "Regular"
    })

    # Missing item weights: item-level median first, then overall median
    item_median_weight = data.groupby("Item_Identifier")["Item_Weight"].transform("median")
    data["Item_Weight"] = data["Item_Weight"].fillna(item_median_weight)
    data["Item_Weight"] = data["Item_Weight"].fillna(data["Item_Weight"].median())

    # Keep missing outlet size as its own category
    data["Outlet_Size"] = data["Outlet_Size"].fillna("Unknown")

    # Derived features
    data["Item_Identifier_Prefix"] = data["Item_Identifier"].str[:2]
    data["Outlet_Age"] = 2013 - data["Outlet_Establishment_Year"]

    outlet_visibility_mean = data.groupby("Outlet_Identifier")["Item_Visibility"].transform("mean")
    data["Item_Visibility_Ratio"] = data["Item_Visibility"] / outlet_visibility_mean.replace(0, np.nan)
    data["Item_Visibility_Ratio"] = data["Item_Visibility_Ratio"].fillna(
        data["Item_Visibility_Ratio"].median()
    )

    # Keep identifiers as categorical features, but remove full item ID to avoid very sparse categories.
    data = data.drop(columns=["Item_Identifier"])
    return data


In [ ]:
# Separate predictors and target
X = df.drop(columns=[target_var])
y = df[target_var]

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.20, random_state=42
)

X_train_p = prepare_features(X_train)
X_valid_p = prepare_features(X_valid)

categorical_features = X_train_p.select_dtypes(include=["object", "category"]).columns.tolist()
print("Training rows:", X_train_p.shape[0])
print("Validation rows:", X_valid_p.shape[0])
print("Categorical features:", categorical_features)
print("Remaining missing training values:", int(X_train_p.isnull().sum().sum()))


## Model training and validation

CatBoost is a gradient-boosted tree model that natively handles categorical features and is a strong fit for this mixed numeric/categorical dataset. Early stopping selects the iteration count based on validation RMSE, helping reduce overfitting.

The validation split is held out from model fitting. RMSE is the primary metric because it is the competition's evaluation metric.


In [ ]:
# Train CatBoost with early stopping
cat_model = CatBoostRegressor(
    iterations=1800,
    depth=6,
    learning_rate=0.035,
    loss_function="RMSE",
    eval_metric="RMSE",
    l2_leaf_reg=5,
    random_seed=42,
    verbose=200,
    allow_writing_files=False
)

cat_model.fit(
    X_train_p, y_train,
    cat_features=categorical_features,
    eval_set=(X_valid_p, y_valid),
    early_stopping_rounds=120,
    verbose=200
)

valid_pred = cat_model.predict(X_valid_p)
rmse = mean_squared_error(y_valid, valid_pred) ** 0.5
mae = mean_absolute_error(y_valid, valid_pred)
r2 = r2_score(y_valid, valid_pred)

print(f"Validation RMSE: {rmse:,.2f}")
print(f"Validation MAE:  {mae:,.2f}")
print(f"Validation R²:   {r2:.4f}")
print("Best iteration:", cat_model.best_iteration_)


In [ ]:
# Actual vs predicted values
plt.figure(figsize=(7, 7))
sns.scatterplot(x=y_valid, y=valid_pred, alpha=0.45)
max_value = max(float(y_valid.max()), float(np.max(valid_pred)))
plt.plot([0, max_value], [0, max_value], linestyle="--")
plt.xlabel("Actual sales")
plt.ylabel("Predicted sales")
plt.title("Validation: Actual vs Predicted Sales")
plt.tight_layout()
plt.show()

residuals = y_valid - valid_pred
plt.figure(figsize=(9, 4))
sns.histplot(residuals, bins=40, kde=True)
plt.title("Validation Residual Distribution (Actual - Predicted)")
plt.xlabel("Residual")
plt.tight_layout()
plt.show()


In [ ]:
# Feature importance
importance = pd.Series(
    cat_model.get_feature_importance(),
    index=X_train_p.columns
).sort_values(ascending=False)

display(importance.to_frame("Importance"))
plt.figure(figsize=(9, 6))
importance.head(15).sort_values().plot(kind="barh")
plt.title("Top 15 Feature Importances")
plt.tight_layout()
plt.show()


## Generalization checks

A single validation split is useful but can depend on the random split. The following optional 5-fold cross-validation gives a more stable estimate of performance. It takes longer to run. Fold-level feature preparation is performed independently to reduce leakage.


In [ ]:
# Optional 5-fold cross-validation
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_rmse = []

for fold, (train_idx, valid_idx) in enumerate(kf.split(X), start=1):
    X_fold_train = prepare_features(X.iloc[train_idx])
    X_fold_valid = prepare_features(X.iloc[valid_idx])
    y_fold_train = y.iloc[train_idx]
    y_fold_valid = y.iloc[valid_idx]

    cats = X_fold_train.select_dtypes(include=["object", "category"]).columns.tolist()
    fold_model = CatBoostRegressor(
        iterations=260,
        depth=6,
        learning_rate=0.035,
        loss_function="RMSE",
        l2_leaf_reg=5,
        random_seed=42,
        verbose=False,
        allow_writing_files=False
    )
    fold_model.fit(X_fold_train, y_fold_train, cat_features=cats, verbose=False)
    fold_pred = fold_model.predict(X_fold_valid)
    fold_score = mean_squared_error(y_fold_valid, fold_pred) ** 0.5
    cv_rmse.append(fold_score)
    print(f"Fold {fold}: RMSE = {fold_score:,.2f}")

print(f"Mean CV RMSE: {np.mean(cv_rmse):,.2f}")
print(f"Std. dev. CV RMSE: {np.std(cv_rmse):,.2f}")


## Fit on all training data and create submission

Put `test.csv` in the same folder as this notebook. The output has exactly the three required columns: `Item_Identifier`, `Outlet_Identifier`, and `Item_Outlet_Sales`.

The final model uses the validation-selected iteration count when available. If `test.csv` is not present, the cell safely skips submission creation.


In [ ]:
import os

test_path = "test.csv"

if os.path.exists(test_path):
    test_df = pd.read_csv(test_path)
    test_features = prepare_features(test_df)

    final_iterations = cat_model.best_iteration_ + 1 if cat_model.best_iteration_ is not None else 260
    final_model = CatBoostRegressor(
        iterations=final_iterations,
        depth=6,
        learning_rate=0.035,
        loss_function="RMSE",
        l2_leaf_reg=5,
        random_seed=42,
        verbose=False,
        allow_writing_files=False
    )

    full_features = prepare_features(X)
    final_cats = full_features.select_dtypes(include=["object", "category"]).columns.tolist()
    final_model.fit(full_features, y, cat_features=final_cats, verbose=False)

    test_predictions = final_model.predict(test_features)
    # Sales cannot be negative; clip only at zero
    test_predictions = np.maximum(test_predictions, 0)

    submission = pd.DataFrame({
        "Item_Identifier": test_df["Item_Identifier"],
        "Outlet_Identifier": test_df["Outlet_Identifier"],
        "Item_Outlet_Sales": test_predictions
    })
    submission.to_csv("submission.csv", index=False)
    print("Saved submission.csv")
    display(submission.head())
    print("Submission shape:", submission.shape)
else:
    print("test.csv was not found. Add it to this folder and rerun this cell to create submission.csv.")


## Notes and next improvements

1. Compare your validation RMSE and 5-fold CV RMSE. A large gap can signal an unstable split or overfitting.
2. Do not repeatedly tune against the same validation set without cross-validation; this can overfit the validation set.
3. If allowed by your course setup, compare CatBoost with LightGBM or XGBoost using the same split and metric.
4. No model can guarantee the lowest unseen-test RMSE. The validation/CV scores estimate generalization, and the final leaderboard score may differ.
